# Knowledge Distillation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS-7150-DL/Distillation/blob/main/scripts/run_colab.ipynb)

This notebook runs knowledge distillation experiments on Google Colab.

**Setup (one-time):** Add tokens to Colab Secrets (🔑 in left sidebar):
1. `GITHUB_TOKEN`: https://github.com/settings/tokens → Generate new token (classic) → Select `repo` scope
2. `HF_TOKEN`: https://huggingface.co/settings/tokens → Create new token (only needed for Llama models)
3. `WANDB_API_KEY`: https://wandb.ai/authorize (optional, for experiment tracking)

**Hardware:** A100 GPU recommended (T4/V100 also work)

**Quick Configs (~30-60 min):**
- BERT → DistilBERT on MNLI
- BERT → DistilBERT on QQP
- Llama 3B → 1B on translation

**Full Configs (~2-8 hours):**
- Use the non-quick versions for paper reproduction

## 1. Setup Environment

In [ ]:
# Mount Google Drive (saves checkpoints even if runtime disconnects)
import os

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
DRIVE_MOUNTED = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = True
        OUTPUT_BASE = "/content/drive/MyDrive/distillation"
        print("Google Drive mounted successfully")
    except Exception as e:
        print(f"Drive mount failed (expected in VS Code + Colab runtime): {e}")
        print("Falling back to local outputs")
        DRIVE_MOUNTED = False
        OUTPUT_BASE = "./outputs"
        os.makedirs(OUTPUT_BASE, exist_ok=True)
else:
    print("Running locally - skipping drive mount")
    OUTPUT_BASE = "../outputs"
    os.makedirs(OUTPUT_BASE, exist_ok=True)

print(f"Output directory: {OUTPUT_BASE}")

In [ ]:
# Keep-alive script (only needed in Colab - prevents disconnection due to inactivity)
if IN_COLAB:
    from IPython.display import Javascript
    display(Javascript('''
        setInterval(() => {
            console.log("Keep alive: " + new Date());
            document.querySelector("colab-connect-button")?.click();
        }, 60000);
    '''))
    print("Keep-alive script running (check browser console)")
else:
    print("Running locally - keep-alive not needed")

In [ ]:
# Clone repository (only in Colab)
if IN_COLAB:
    import os
    if os.path.exists('/content/Distillation'):
        print("Repository already exists, pulling latest changes...")
        %cd /content/Distillation
        !git pull
    else:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/CS-7150-DL/Distillation.git /content/Distillation
        %cd /content/Distillation
else:
    print("Running locally - using current repository")

In [ ]:
# Install dependencies (always run after clone/pull)
if IN_COLAB:
    !pip install -q uv
    !uv pip install -q -e . --system
    print("Dependencies installed successfully")
else:
    print("Running locally - assuming dependencies are already installed")
    print("If not, run: pip install -e . from the project root")

## 2. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
# Verify environment and device
import os
import torch

def detect_environment():
    if os.environ.get("SLURM_JOB_ID"):
        return "slurm"
    elif os.environ.get("COLAB_GPU"):
        return "colab"
    return "local"

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

print(f"Environment: {detect_environment()}")
print(f"Device: {get_device()}")

## 3. Login to Services

In [ ]:
# W&B login (for experiment tracking)
# Uses WANDB_API_KEY from Colab Secrets, or set WANDB_MODE=disabled to skip
import os
if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
        print("W&B API key loaded from secrets")
    except:
        print("No WANDB_API_KEY in secrets - set WANDB_MODE=disabled to skip W&B")
        os.environ['WANDB_MODE'] = 'disabled'

In [ ]:
# HuggingFace login (only needed for Llama models)
# Skip this cell if running BERT experiments
if IN_COLAB:
    try:
        from google.colab import userdata
        from huggingface_hub import login
        login(token=userdata.get('HF_TOKEN'))
        print("HuggingFace login successful")
    except Exception as e:
        print(f"HF login failed: {e}")
        print("Add HF_TOKEN to Colab Secrets for Llama models")

---
## 4. Run Training

Choose ONE of the following experiments to run.

### Option A: BERT → DistilBERT on MNLI (Recommended Start)

| Version | Time | Expected Accuracy |
|---------|------|-------------------|
| Quick | ~30-45 min | ~0.75-0.78 |
| Full | ~2-3 hours | ~0.817 |

In [ ]:
# MNLI - Quick version (~30-45 min)
!python train.py \
    --config configs/bert_distilbert_mnli_quick.yaml \
    --output_dir {OUTPUT_BASE}/mnli_quick

In [ ]:
# MNLI - Full version (~2-3 hours)
!python train.py \
    --config configs/bert_distilbert_mnli.yaml \
    --output_dir {OUTPUT_BASE}/mnli_full

### Option B: BERT → DistilBERT on QQP

| Version | Time | Expected Accuracy |
|---------|------|-------------------|
| Quick | ~30-45 min | ~0.85-0.88 |
| Full | ~2-3 hours | ~0.900 |

In [ ]:
# QQP - Quick version (~30-45 min)
!python train.py \
    --config configs/bert_distilbert_qqp_quick.yaml \
    --output_dir {OUTPUT_BASE}/qqp_quick

In [ ]:
# QQP - Full version (~2-3 hours)
!python train.py \
    --config configs/bert_distilbert_qqp.yaml \
    --output_dir {OUTPUT_BASE}/qqp_full

### Option C: Llama 3B → 1B Translation (Requires HF Login)

| Version | Time | Expected BLEU |
|---------|------|---------------|
| Quick | ~45-60 min | ~15-20 |
| Full | ~6-8 hours | ~27 |

In [ ]:
# Translation - Quick version (~45-60 min)
!python train.py \
    --config configs/llama_3b_1b_translation_quick.yaml \
    --output_dir {OUTPUT_BASE}/translation_quick

In [ ]:
# Translation - Full version (~6-8 hours)
!python train.py \
    --config configs/llama_3b_1b_translation.yaml \
    --output_dir {OUTPUT_BASE}/translation_full

---
## 5. View Results

In [ ]:
# View final metrics
import json
from pathlib import Path

# Uses OUTPUT_BASE from earlier cell
output_dir = f"{OUTPUT_BASE}/mnli_quick"

trainer_state = Path(output_dir) / "trainer_state.json"

if trainer_state.exists():
    with open(trainer_state) as f:
        state = json.load(f)
    
    print("=" * 50)
    print("TRAINING RESULTS")
    print("=" * 50)
    
    if 'log_history' in state:
        # Get final eval metrics
        eval_logs = [log for log in state['log_history'] if 'eval_loss' in log]
        if eval_logs:
            final_eval = eval_logs[-1]
            print("\nFinal Evaluation Metrics:")
            for key, value in final_eval.items():
                if key.startswith('eval_'):
                    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
else:
    print(f"No trainer state found at {trainer_state}")
    print("Training may still be in progress or output_dir is incorrect.")

In [ ]:
# List saved checkpoints
!ls -la {OUTPUT_BASE}/mnli_quick/

---
## 6. Download Model (Optional)

In [ ]:
# Download trained model (only needed in Colab with drive mounted)
if IN_COLAB and DRIVE_MOUNTED:
    from google.colab import files
    import shutil

    output_dir = f"{OUTPUT_BASE}/mnli_quick"
    shutil.make_archive("/content/trained_model", 'zip', output_dir)
    files.download("/content/trained_model.zip")
else:
    print(f"Model saved at: {OUTPUT_BASE}/mnli_quick")

---
## Troubleshooting

**Colab disconnects:**
- Make sure keep-alive cell ran
- On your laptop: run `caffeinate -dims` in terminal
- Checkpoints saved to Google Drive will persist

**Out of memory:**
- Restart runtime and try again
- Use quick configs (smaller batch sizes)

**W&B issues:**
- Set `WANDB_MODE=disabled` to skip W&B:
  ```python
  import os
  os.environ['WANDB_MODE'] = 'disabled'
  ```